In [ ]:
import anndata as ad

INPUT = "Multiome_scc_IDEAS_input.h5ad"

adata = ad.read_h5ad(INPUT)

print(adata)
print("Shape:", adata.shape)

AnnData object with n_obs × n_vars = 14302 × 26408
Shape: (14302, 26408)


In [ ]:
import pandas as pd

emt_annotation = pd.read_csv(
    "emt_state_annotation.csv",
    index_col="cell_id"
)

print(emt_annotation.shape)
print(emt_annotation.head())

(14371, 1)
                                   emt_state
cell_id                                     
MU1_AAACCAACATTAAAGG-1  Early hybrid (EMT-I)
MU1_AAACCGCGTATTGTGG-1                Hybrid
MU1_AAACCGGCAGTTTGGC-1            Epithelial
MU1_AAACGCGCAGCAATAA-1            Epithelial
MU1_AAACGTACAAGCTAAA-1            Epithelial


In [ ]:
common_cells = adata.obs_names.intersection(emt_annotation.index)

print("Cells in adata:", adata.n_obs)
print("Cells in CSV:", len(emt_annotation))
print("Common cells:", len(common_cells))

Cells in adata: 14302
Cells in CSV: 14371
Common cells: 14302


In [ ]:
# Transfer EMT annotation using cell IDs
adata.obs["emt_state"] = emt_annotation.loc[
    adata.obs_names,
    "emt_state"
].values

In [ ]:
print(adata.obs["emt_state"].value_counts(dropna=False))

emt_state
Epithelial               5827
Early hybrid (EMT-I)     2472
Late EMT                 2451
Early hybrid (EMT-II)    1830
Hybrid                   1722
Name: count, dtype: int64


In [ ]:
print("Missing annotations:", adata.obs["emt_state"].isna().sum())

Missing annotations: 0


In [ ]:
missing_cells = emt_annotation.index.difference(adata.obs_names)

print("Missing cells:", len(missing_cells))

print(
    emt_annotation.loc[
        missing_cells,
        "emt_state"
    ].value_counts()
)

Missing cells: 69
emt_state
Low quality    69
Name: count, dtype: int64


In [ ]:
adata.obs["emt_state"] = emt_annotation.loc[
    adata.obs_names,
    "emt_state"
].values

In [ ]:
print(adata.obs["emt_state"].value_counts())
print("\nMissing:", adata.obs["emt_state"].isna().sum())

emt_state
Epithelial               5827
Early hybrid (EMT-I)     2472
Late EMT                 2451
Early hybrid (EMT-II)    1830
Hybrid                   1722
Name: count, dtype: int64

Missing: 0


In [ ]:
adata.write_h5ad("adata_with_emt_state.h5ad")

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

# IDEAS source
sys.path.insert(0, "/home/csb/sampoorn/IDEAS")

from computeID.functions_IDEAS import compute_ID


# ---------------------------------------------------------
# Balanced global 2NN ID
# ---------------------------------------------------------

# Smallest population = Hybrid = 1722 cells
# 80% of 1722 = 1377.6
# Therefore every state uses 1377 cells per replicate

id_reps = compute_ID(
    adata,
    group="emt_state",
    method="2nn",
    sample_size=1377,
    n_samples=30,
    normalization=True,
    norm_sum=1e6,
    full_output=True,
    id_score=False
)

print(id_reps)
print("Shape:", id_reps.shape)


# ---------------------------------------------------------
# Box plot
# ---------------------------------------------------------

order = [
    "Epithelial",
    "Early hybrid (EMT-I)",
    "Early hybrid (EMT-II)",
    "Hybrid",
    "Late EMT"
]

plt.figure(figsize=(9, 5))

plt.boxplot(
    [
        id_reps[state].dropna().values
        for state in order
    ],
    labels=order,
    showfliers=False
)


# ---------------------------------------------------------
# Overlay the 30 individual balanced estimates
# ---------------------------------------------------------

rng = np.random.default_rng(42)

for i, state in enumerate(order, start=1):

    values = id_reps[state].dropna().values

    x = (
        i
        + 0.04 * rng.uniform(-1, 1, len(values))
    )

    plt.scatter(
        x,
        values,
        s=20,
        alpha=0.6
    )


# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

plt.ylabel("Global Intrinsic Dimension")
plt.xlabel("EMT State")

plt.xticks(
    range(1, len(order) + 1),
    order,
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

# IDEAS source
sys.path.insert(0, "/home/csb/sampoorn/IDEAS")

from computeID.functions_IDEAS import compute_ID


# ---------------------------------------------------------
# Balanced global 2NN ID
# ---------------------------------------------------------

# Smallest population = Hybrid = 1722 cells
# 80% of 1722 = 1377.6
# Therefore every state uses 1377 cells per replicate

id_reps = compute_ID(
    adata,
    group="emt_state",
    method="2nn",
    sample_size=1377,
    n_samples=100,
    normalization=True,
    norm_sum=1e6,
    full_output=True,
    id_score=False
)

print(id_reps)
print("Shape:", id_reps.shape)


# ---------------------------------------------------------
# Box plot
# ---------------------------------------------------------

order = [
    "Epithelial",
    "Early hybrid (EMT-I)",
    "Early hybrid (EMT-II)",
    "Hybrid",
    "Late EMT"
]

plt.figure(figsize=(9, 5))

plt.boxplot(
    [
        id_reps[state].dropna().values
        for state in order
    ],
    labels=order,
    showfliers=False
)


# ---------------------------------------------------------
# Overlay the 30 individual balanced estimates
# ---------------------------------------------------------

rng = np.random.default_rng(42)

for i, state in enumerate(order, start=1):

    values = id_reps[state].dropna().values

    x = (
        i
        + 0.04 * rng.uniform(-1, 1, len(values))
    )

    plt.scatter(
        x,
        values,
        s=20,
        alpha=0.6
    )


# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

plt.ylabel("Global Intrinsic Dimension")
plt.xlabel("EMT State")

plt.xticks(
    range(1, len(order) + 1),
    order,
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()

Computing Intrinsic Dimension:   0%|          | 0/5 [00:00<?, ?it/s]

Computing ID for emt_state Early hybrid (EMT-I):


Computing Intrinsic Dimension:  20%|██        | 1/5 [02:23<09:35, 143.90s/it]

Computing ID for emt_state Early hybrid (EMT-II):


Computing Intrinsic Dimension:  40%|████      | 2/5 [04:55<07:25, 148.35s/it]

...

Computing ID for emt_state Hybrid:


Computing Intrinsic Dimension:  80%|████████  | 4/5 [10:15<02:32, 152.63s/it]

Computing ID for emt_state Late EMT:


Computing Intrinsic Dimension: 100%|██████████| 5/5 [12:20<00:00, 148.03s/it]


     Early hybrid (EMT-I)  Early hybrid (EMT-II)  Epithelial     Hybrid  \
1               31.012510              27.935738   30.005588  27.242897   
2               31.505294              26.670620   29.116166  27.108618   
3               30.778027              27.842415   29.958178  25.488347   
4               31.803758              27.246009   28.533514  25.543834   
5               30.934253              29.196059   27.576493  25.809576   
..                    ...                    ...         ...        ...   
96              30.430982              27.810132   28.708787  26.123165   
97              31.957917              27.648280   29.532293  26.065167   
98              31.042036              26.720587   28.877535  25.855148   
99              31.125990              28.575656   28.157289  25.825529   
100             32.188472              28.066208   28.736872  26.322686   

      Late EMT  
1    26.018075  
2    25.943753  
3    26.357583  
4    27.338054  
5    25.380280

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal, rankdata, norm
from statsmodels.stats.multitest import multipletests
from itertools import combinations


# ============================================================
# SETTINGS
# ============================================================

order = [
    "Epithelial",
    "Early hybrid (EMT-I)",
    "Early hybrid (EMT-II)",
    "Hybrid",
    "Late EMT"
]


# ============================================================
# EXTRACT ID REPLICATES
# ============================================================

data = {
    state: id_reps[state].dropna().values
    for state in order
}

print("Number of ID replicates:")
for state in order:
    print(f"{state}: {len(data[state])}")


# ============================================================
# 1. OMNIBUS KRUSKAL-WALLIS TEST
# ============================================================

H, p_kw = kruskal(
    *[data[state] for state in order]
)

print("\n==============================================")
print("KRUSKAL-WALLIS TEST")
print("==============================================")
print(f"H statistic = {H:.4f}")
print(f"p-value     = {p_kw:.4e}")


# ============================================================
# 2. DUNN'S POST-HOC TEST
# ============================================================

all_values = np.concatenate(
    [data[state] for state in order]
)

groups = np.concatenate([
    np.repeat(state, len(data[state]))
    for state in order
])


# Rank all observations together
ranks = rankdata(all_values)

rank_df = pd.DataFrame({
    "group": groups,
    "rank": ranks
})

N = len(all_values)


# Mean rank for each state
mean_ranks = (
    rank_df
    .groupby("group")["rank"]
    .mean()
)


# Number of observations per state
n_groups = {
    state: len(data[state])
    for state in order
}


# ============================================================
# TIE CORRECTION
# ============================================================

_, tie_counts = np.unique(
    all_values,
    return_counts=True
)

tie_term = np.sum(
    tie_counts**3 - tie_counts
)

tie_correction = (
    1 -
    tie_term / (N**3 - N)
)


# ============================================================
# PAIRWISE DUNN TESTS
# ============================================================

results = []

for g1, g2 in combinations(order, 2):

    n1 = n_groups[g1]
    n2 = n_groups[g2]

    rank_difference = (
        mean_ranks[g1] -
        mean_ranks[g2]
    )

    SE = np.sqrt(
        (
            N * (N + 1) / 12
        ) *
        (
            1 / n1 +
            1 / n2
        ) *
        tie_correction
    )

    z = rank_difference / SE

    p = 2 * norm.sf(abs(z))

    results.append({
        "group1": g1,
        "group2": g2,
        "z": z,
        "p_uncorrected": p
    })


dunn_df = pd.DataFrame(results)


# ============================================================
# 3. BENJAMINI-HOCHBERG FDR
# ============================================================

dunn_df["p_FDR"] = multipletests(
    dunn_df["p_uncorrected"],
    method="fdr_bh"
)[1]


# ============================================================
# 4. SIGNIFICANCE LABELS
# ============================================================

def significance_label(p):

    if p < 0.0001:
        return "****"

    elif p < 0.001:
        return "***"

    elif p < 0.01:
        return "**"

    elif p < 0.05:
        return "*"

    else:
        return "ns"


dunn_df["significance"] = (
    dunn_df["p_FDR"]
    .apply(significance_label)
)


# ============================================================
# 5. PRINT COMPLETE STATISTICS
# ============================================================

print("\n==============================================")
print("DUNN POST-HOC TEST — BH FDR CORRECTED")
print("==============================================")

print(
    dunn_df.to_string(index=False)
)


# ============================================================
# 6. CREATE P-VALUE MATRIX
# ============================================================

p_matrix = pd.DataFrame(
    np.nan,
    index=order,
    columns=order
)

sig_matrix = pd.DataFrame(
    "",
    index=order,
    columns=order
)


for _, row in dunn_df.iterrows():

    g1 = row["group1"]
    g2 = row["group2"]

    p_matrix.loc[g2, g1] = row["p_FDR"]

    sig_matrix.loc[g2, g1] = (
        significance_label(row["p_FDR"])
    )


# ============================================================
# 7. PLOT — DUNN FDR HEATMAP
# ============================================================

plt.figure(figsize=(10, 8))

mask = np.triu(
    np.ones_like(
        p_matrix.astype(float),
        dtype=bool
    )
)

sns.heatmap(
    p_matrix.astype(float),
    mask=mask,
    annot=False,
    cmap="viridis_r",
    vmin=0,
    vmax=0.05,
    linewidths=2,
    linecolor="white",
    cbar_kws={
        "label": "Dunn test BH-adjusted p-value"
    }
)


# Add p-values + significance stars
for i in range(len(order)):

    for j in range(len(order)):

        if i > j:

            p = p_matrix.iloc[i, j]

            sig = sig_matrix.iloc[i, j]

            if pd.isna(p):
                continue

            if p < 0.001:
                text = f"{p:.2e}\n{sig}"
            else:
                text = f"{p:.3f}\n{sig}"

            plt.text(
                j + 0.5,
                i + 0.5,
                text,
                ha="center",
                va="center",
                fontsize=10
            )


plt.title(
    "IDEAS Global Intrinsic Dimension: Pairwise Dunn Comparisons",
    fontsize=14
)

plt.xlabel("")
plt.ylabel("")

plt.xticks(
    np.arange(len(order)) + 0.5,
    order,
    rotation=30,
    ha="right"
)

plt.yticks(
    np.arange(len(order)) + 0.5,
    order,
    rotation=0
)

plt.tight_layout()
plt.show()

Number of ID replicates:
Epithelial: 100
Early hybrid (EMT-I): 100
Early hybrid (EMT-II): 100
Hybrid: 100
Late EMT: 100

KRUSKAL-WALLIS TEST
H statistic = 414.8294
p-value     = 1.7373e-88

DUNN POST-HOC TEST — BH FDR CORRECTED
               group1                group2         z  p_uncorrected        p_FDR significance
           Epithelial  Early hybrid (EMT-I) -5.638479   1.715591e-08 2.450845e-08         ****
           Epithelial Early hybrid (EMT-II)  4.155081   3.251729e-05 3.613033e-05         ****
           Epithelial                Hybrid 12.125103   7.776272e-34 2.592091e-33         ****
           Epithelial              Late EMT  9.700572   2.998145e-22 5.996290e-22         ****
 Early hybrid (EMT-I) Early hybrid (EMT-II)  9.793559   1.199959e-22 2.999899e-22         ****
 Early hybrid (EMT-I)                Hybrid 17.763581   1.353185e-70 1.353185e-69         ****
 Early hybrid (EMT-I)              Late EMT 15.339050   4.192854e-53 2.096427e-52         ****
Early hybrid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

order = [
    "Epithelial",
    "Early hybrid (EMT-I)",
    "Early hybrid (EMT-II)",
    "Hybrid",
    "Late EMT"
]

plt.figure(figsize=(9, 5))

# Box plots
plt.boxplot(
    [
        id_reps[state].dropna().values
        for state in order
    ],
    labels=order,
    showfliers=False
)

# Overlay individual replicate estimates
rng = np.random.default_rng(42)

for i, state in enumerate(order, start=1):

    values = id_reps[state].dropna().values

    x = (
        i
        + 0.04 * rng.uniform(-1, 1, len(values))
    )

    plt.scatter(
        x,
        values,
        s=20,
        alpha=0.6
    )

# Labels
plt.ylabel("ID Score")
plt.xlabel("EMT State")

plt.xticks(
    range(1, len(order) + 1),
    order,
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
# If needed, run once first:
%pip install gseapy scanpy

import gseapy as gp
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse

# Set True only if the new adata.X contains raw UMI counts.
# Keep False if adata.X is already normalized and log1p-transformed.
INPUT_IS_RAW_COUNTS = False
CHUNK_SIZE = 2000

# Mouse KS epithelial signature
KS_EPI_MOUSE = """
Cdh1 Agr2 Epcam Krt19 Rab25 Tacstd2 Galnt3 Fxyd3 Spint2 Tmem30b Scnn1a St14
Esrp1 S100a14 Cldn7 Erbb3 Rbm47 Spint1 Elf3 Cldn13 Prss8 Sh3yl1 Ehf Lcn2 Jup
Vamp8 Krt8 Inava Krt7 Dsp Cds1 Itgb4 Tmprss4 Lsr Sorl1 Grhl2 Ppl AA986860
Tspan1 Map7 Slpi Tox3 Arhgap8 F11r Lad1 Gpx2 Ctsh Adgrg1 Fa2h Klf5 Areg Krt18
Scel Cdh3 Ugt1a1 Mpzl2 Crybg1 Ovol2 Llgl2 Esrp2 Myo5c Ddr1 Vgll1 Irf6 Sfn
Tspan13 Kcnk1 Myo1d Pkp3 Itgb6 Ly75 Mapk13 Ttc39a Elmo3 Ceacam1 Dtx4 Erbb2
Rab11fip1 Atp2c2 Mst1r Ap1m2 Tgfa Myo6 Ptk6 Oas1g Fbp1 Aqp3 Cblc Epha1 Bspry
Eps8l1 Grb7 0610040J01Rik Klk6 Tjp2 Pls1 Dennd2d Eps8l2 Il20ra Hes1 Il1rn
Exph5 Arhgdib Misp Camk2n1 Hpgd Syngr2 Perp Mansc1 Dsc2 Pof1b Serinc5 Bik
Anxa9 Mall Epn3 Stap2 Foxa1 Pycard Slc37a1 Ank3 Tspan15 Hnmt Abcc3 Sdc1 Ckmt1
Tob1 B3gnt3 Tmc6 Cd9 Adap1 Atp1b1 Shank2 Cyb561 Ermp1 Rab20 Myh14 Capn1
Aldh3b2 Trim31 Arap2 Ssh3 Ica1 Arhgef5 Alox5 Rhod Tmprss2 Mtus1 Cyp4f18
Ppfibp2 Rabgap1l Plxnb2 Mgst2 Evpl Cd46 Krt15 Cnksr1 Blnk Comt Anxa4 Tnfsf13
Ocln Nherf1 Xbp1
""".split()

# Mouse KS mesenchymal signature
KS_MES_MOUSE = """
Vim Zeb1 Emp3 Sacs Axl Loxl2 Sparc Fhl4 Fermt2 Tuba1a Tmem158 Cald1 Lgals1
Pmp22 Msn Glyr1 Map1b Ap1s2 Gja1 Dennd5a Fam216a Tpm2 Tubb6 Srpx Ank2 Chn1
Sh2b3 P3h1 Etv1 Sobp Akap12 Tgfb1i1 Serpine1 Soat1 Col5a2 Lhfpl6 Cep170 Popdc3
Trpc1 Poglut2 Myl9 Bag2 Fstl1 Mxra7 Gfpt2 Reck Tmeff1 Cavin1
""".split()

# Normalize only raw-count data; do not run this twice.
if INPUT_IS_RAW_COUNTS:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

available_genes = set(adata.var_names.astype(str))
epi_found = [g for g in KS_EPI_MOUSE if g in available_genes]
mes_found = [g for g in KS_MES_MOUSE if g in available_genes]

print(f"KS-Epi genes found: {len(epi_found)}/{len(KS_EPI_MOUSE)}")
print(f"KS-Mes genes found: {len(mes_found)}/{len(KS_MES_MOUSE)}")

if len(epi_found) < 20 or len(mes_found) < 5:
    raise ValueError(
        "Too few genes matched. adata.var_names should contain mouse gene "
        "symbols (e.g. Cdh1, Epcam, Vim), not Ensembl IDs."
    )

gene_sets = {"KS_Epi": KS_EPI_MOUSE, "KS_Mes": KS_MES_MOUSE}
all_scores = []

for start in range(0, adata.n_obs, CHUNK_SIZE):
    stop = min(start + CHUNK_SIZE, adata.n_obs)
    X = adata.X[start:stop]

    if issparse(X):
        X = X.T.toarray()
    else:
        X = np.asarray(X).T

    expression = pd.DataFrame(
        X,
        index=adata.var_names.astype(str),
        columns=adata.obs_names[start:stop].astype(str),
    ).groupby(level=0, sort=False).mean()

    result = gp.ssgsea(
        data=expression,
        gene_sets=gene_sets,
        sample_norm_method="rank",
        permutation_num=0,
        min_size=5,
        max_size=500,
        outdir=None,
        no_plot=True,
        threads=1,
    )

    scores_chunk = result.res2d.pivot(
        index="Name",
        columns="Term",
        values="ES",
    ).reindex(adata.obs_names[start:stop].astype(str))

    all_scores.append(scores_chunk)
    print(f"Scored cells {start + 1:,}–{stop:,} of {adata.n_obs:,}")

scores = pd.concat(all_scores).reindex(adata.obs_names.astype(str))

# Add scores directly to this new adata object
adata.obs["KS_Epi_ssGSEA"] = scores["KS_Epi"].to_numpy()
adata.obs["KS_Mes_ssGSEA"] = scores["KS_Mes"].to_numpy()

print("\nDone:")
display(adata.obs[["KS_Epi_ssGSEA", "KS_Mes_ssGSEA"]].head())

/bin/bash: /home/csb/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Obtaining dependency information for gseapy from https://files.pythonhosted.org/packages/f4/a3/ca21623386d4af866a0a11c8f6a8162266347c566ef68d2dd3101b2acd8d/gseapy-1.3.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.0/693.0 kB 6.6 MB/s eta 0:00:00a 0:00:01
DEPRECATION: free 0.0.2 has a non-standard dependency specifier django>=1.11.0<1.12. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of free or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: free 0.0.2 has a non-standard dependency specifier jupyter>=1.0.0<1.1.0. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of free or co

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Existing annotations and scores
state_col = "emt_state"
epi_col = "KS_Epi_ssGSEA"
mes_col = "KS_Mes_ssGSEA"

state_order = [
    "Epithelial",
    "Early hybrid (EMT-I)",
    "Early hybrid (EMT-II)",
    "Hybrid",
    "Late EMT",
]

# Keep only cells with valid scores and one of the five EMT annotations
plot_df = (
    adata.obs[[state_col, epi_col, mes_col]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .loc[lambda x: x[state_col].isin(state_order)]
    .copy()
)

# Pearson correlation across all cells
pearson_r, pearson_p = pearsonr(
    plot_df[epi_col],
    plot_df[mes_col],
)

print(f"Pearson correlation: r = {pearson_r:.3f}")
print(f"P-value: {pearson_p:.3e}")
print(f"Cells plotted: {len(plot_df):,}")
display(plot_df[state_col].value_counts().reindex(state_order))

# Colours for the existing ordered EMT states
palette = {
    "Epithelial": "#277DA1",             # blue
    "Early hybrid (EMT-I)": "#56B4E9",   # light blue
    "Early hybrid (EMT-II)": "#F4A261",  # orange
    "Hybrid": "#7B2CBF",                 # purple
    "Late EMT": "#D62828",               # red
}

sns.set_theme(style="white", context="paper")

fig, ax = plt.subplots(figsize=(8.5, 6.5))

sns.scatterplot(
    data=plot_df,
    x=epi_col,
    y=mes_col,
    hue=state_col,
    hue_order=state_order,
    palette=palette,
    s=12,
    alpha=0.45,
    linewidth=0,
    ax=ax,
)

# Overall linear regression line
x = plot_df[epi_col].to_numpy()
y = plot_df[mes_col].to_numpy()
slope, intercept = np.polyfit(x, y, 1)

x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(
    x_line,
    slope * x_line + intercept,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="Overall trend",
)

p_text = "p < 1e-300" if pearson_p == 0 else f"p = {pearson_p:.2e}"

ax.text(
    0.03,
    0.97,
    f"Pearson r = {pearson_r:.3f}\n{p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
)

ax.set_xlabel("KS-Epi ssGSEA score", fontsize=12)
ax.set_ylabel("KS-Mes ssGSEA score", fontsize=12)
ax.set_title("KS-Epi versus KS-Mes across EMT States", fontsize=13)

ax.legend(title="EMT state", frameon=False, loc="best")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Pearson correlation: r = -0.074
P-value: 5.206e-19
Cells plotted: 14,302
